In [ ]:
#Step-1: get the data from api
import requests
import json

response = requests.get("https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US")
data = json.loads(response.text)
result = data['results']

total_page = data['total_pages']
print(total_page)
print(result)
print(data)

In [ ]:
from ast import Return
#If we get all the record from total page
movie_data = []
genre_data = []
##Take the genre from this api link
res = requests.get("https://api.themoviedb.org/3/genre/movie/list?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US")
data = json.loads(res.text)
print(data['genres'])
for genre in data['genres']:
  genre_data.append({
      "id": genre.get('id'),
      "name": genre.get('name')
  })

for page in range(1, total_page+1):
  if page <= 500:
    api_url = f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page={page}"
    response = requests.get(api_url).json()
    #res = json.loads(response.text)

    for movie in response['results']:
      genre_name = [genre_data[i].get('name') for i in range(0, len(genre_data)) if genre_data[i].get('id') in movie.get('genre_ids')]
      movie_data.append({
        "Name": movie.get('title'),
        "Description": movie.get('overview'),
        "Genere" : ", ".join(genre_name) ##Convert list to string
      })
else:
  Return
print(movie_data)

In [ ]:
##Convert it into dataframe
import pandas as pd

df = pd.DataFrame(movie_data)
df.to_csv("movie_data.csv", index='false', encoding='utf-8')

In [ ]:
## Text preprocessing

#Step-1: Convert it into lower case
df['Name'] = df['Name'].str.lower()
df['Description'] = df['Description'].str.lower()
df['Genere'] = df['Genere'].str.lower()
print(df.head)

In [ ]:
##Step-2 Remove Punctuation
import string
exclude = string.punctuation
def remove_punc(text):
  return text.translate(str.maketrans('', '', exclude))

for col in df.select_dtypes(include='object').columns:
  df[col] = df[col].apply(lambda text: remove_punc(text))

print(df.head)
df['Name'] = df['Name'].apply(lambda text: remove_punc(text))
df['Description'] = df['Description'].apply(lambda text: remove_punc(text))
df['Genere'] = df['Genere'].apply(lambda text: remove_punc(text))

print(df['Name'][1])
print(df['Description'][1])

In [ ]:
##Step-3: Spelling Correction
from textblob import TextBlob

textlb1 = TextBlob(df['Name'][2])
textlb2 = TextBlob(df['Description'][2])
textlb3 = TextBlob(df['Genere'][2])

print(textlb1.correct().string)
print(textlb2.correct().string)
print(textlb3.correct().string)


In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = stopwords.words('english')

def remove_stopword(text):
  # Split the text into words, filter out stopwords, and then join them back
  return " ".join([word for word in text.split() if word not in stop_words])

# Apply stopword removal to all 'object' type columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].apply(lambda text: remove_stopword(text))

print(df['Name'][2])
print(df['Description'][2])
print(df['Genere'][2])

In [ ]:
##Step-5: Tokenization
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

# Tokenize the 'Description' column for all rows
df['tokenized_description'] = df['Description'].apply(word_tokenize)

# Display the original and tokenized descriptions for verification
display(df[['Description', 'tokenized_description']].head())

In [ ]:
##Step-6: Lemmatization
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

for word in df['tokenized_description'][2]:
  print("{0:20}{1:20}".format(word,lemmatizer.lemmatize(word,pos='v')))